# SASHIMI-C: usage walkthrough

Create a small CDM population, compare the established and ITAMAE-backed legacy paths, inspect/export a weighted catalogue, and compute a mass function. Run **Restart Kernel and Run All**. These settings are intentionally small teaching grids, not converged science settings.

## Setup
Use Python 3.12 from this repository's `itamae-migration` checkout. Install the exact ITAMAE revision used by the migration branch, then this package with its demo dependencies. Open the notebook with the `python3` kernel from that environment.


In [ ]:
import tempfile
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from itamae.provenance import source_revision

packages = ['itamae', 'sashimi-c']
display({p: {'version': version(p), 'source_revision': source_revision(p)} for p in packages})
workspace = tempfile.TemporaryDirectory(prefix='sashimi-c-walkthrough-')
work = Path(workspace.name)


## Configure and evolve a small CDM catalogue
Masses are numerical solar masses. Solver and disruption choices are explicit so the comparison is reproducible.


In [ ]:
from sashimi_c import subhalo_properties as LegacyProperties
from sashimi_c_itamae import subhalo_observables, subhalo_properties

parameters = {
    'M0': 1e12, 'redshift': 0.0, 'dz': 0.5, 'zmax': 2.0,
    'N_ma': 24, 'sigmalogc': 0.128, 'N_herm': 3,
    'logmamin': 6.0, 'logmamax': 10.0, 'N_hermNa': 8,
    'Na_model': 3, 'ct_th': 0.0, 'method': 'pert2_shanks',
    'profile_change': True,
}
model = subhalo_properties(physics_mode='legacy')
catalog = model.subhalo_catalog_calc(**parameters)
old = LegacyProperties().subhalo_properties_calc(**parameters)
new = model.subhalo_properties_calc(**parameters)
for established, migrated in zip(old, new):
    np.testing.assert_allclose(established, migrated, rtol=5e-12, atol=0)
consistent = subhalo_properties(physics_mode='consistent').subhalo_catalog_calc(**parameters)
print('legacy / consistent expected counts:', catalog.weight_final.sum(), consistent.weight_final.sum())


## Inspect, select, and plot the weighted catalogue
Rows are quadrature nodes; `weight_final` is the effective population count.


In [ ]:
from itamae.types import WeightedSubhaloCatalog

if not np.all(np.isfinite(catalog.weight_final)) or np.any(catalog.weight_final < 0):
    raise RuntimeError('Invalid catalogue weights')
mass = np.asarray(catalog.columns['m_bound'])
positive = (mass > 0) & (catalog.weight_final > 0)
selected = catalog.select(positive)
edges = np.geomspace(mass[positive].min() * 0.99, mass[positive].max() * 1.01, 16)
counts, edges = selected.weighted_histogram('m_bound', bins=edges)
centres = np.sqrt(edges[:-1] * edges[1:])
density = counts / np.diff(np.log(edges))
plt.loglog(centres, density, marker='o')
plt.xlabel('Bound mass [catalogue mass unit; see metadata]')
plt.ylabel('dN/dln M')
plt.show()
display(catalog.metadata)
print('weighted bound mass:', selected.weighted_sum(selected.columns['m_bound']))


## Export and restore the catalogue


In [ ]:
path = work / 'catalog.npz'
catalog.to_npz(path)
restored = WeightedSubhaloCatalog.from_npz(path)
for name in catalog.columns:
    np.testing.assert_array_equal(restored.columns[name], catalog.columns[name])
np.testing.assert_array_equal(restored.weight_final, catalog.weight_final)
print(path)


## Observable-facing mass functions


In [ ]:
obs_parameters = {('M0_per_Msun' if key == 'M0' else key): value for key, value in parameters.items()}
observables = subhalo_observables(physics_mode='legacy', **obs_parameters)
for evolved in (False, True):
    mass_axis, density = observables.mass_function(evolved=evolved)
    if not np.all(np.isfinite(density)):
        raise RuntimeError('Mass function contains non-finite values')
    plt.loglog(mass_axis, density, label=f'evolved={evolved}')
plt.legend()
plt.xlabel('Mass')
plt.ylabel('Mass function')
plt.show()


## Coverage / limitations

| Area | Demonstrated | Not claimed |
|---|---|---|
| Legacy compatibility | tuple-level comparison on a bounded grid | full scientific validation |
| ITAMAE catalogue | columns, weights, metadata, selection | final frozen API |
| I/O | NPZ round trip | long-term archival format |
| Observables | weighted mass function | convergence / publication settings |
| Physics modes | legacy + consistent execution | preference for a new default |

Prompt-cusp validation, current-main/Picard reconciliation, and scientific one-factor-at-a-time comparisons remain separate migration tasks.
